In [ ]:
# Cross-user results
# user 1: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/office0_lcx_1_cross_user.json
# user 2: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/office0_sf_0_cross_user.json

# Cross-environment results
# env 1: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/hall5_cross_env.json
# env 2: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/office2_0_cross_env.json
# env 3: 

In [ ]:
# use json output to calculate accuracy, precision, recall, f1-score for each model and compare the results
# save acc/pre/recall/f1 in a table into a csv file for further analysis
# save confusion matrix
# example: input is /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/office0_sf_0_cross_user.json
# output to a folder: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta/office0_sf_0_cross_user/

In [4]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Label mapping for posture detection
label_to_text = {
    -1: "unknown",
    0: "absence",
    1: "presence",
    2: "standing",
    3: "sitting_by_bed",
    4: "sitting_on_bed",
    5: "lying_no_cover",
    6: "lying_with_cover"
}
kept_labels = [0, 2, 3, 4, 5, 6]
kept_label_names = [label_to_text[l] for l in kept_labels]

# Input JSON files
json_files = {
    "cross_user": [
        "/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/office0_lcx_1_cross_user.json",
        "/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/office0_sf_0_cross_user.json",
    ],
    "cross_env": [
        "/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/hall5_cross_env.json",
        "/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/previous/office2_0_cross_env.json",
    ]
}

output_base = "/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta"
os.makedirs(output_base, exist_ok=True)

results_summary = []

for exp_type, files in json_files.items():
    for json_path in files:
        if not os.path.exists(json_path):
            print(f"File not found: {json_path}")
            continue

        with open(json_path, 'r') as f:
            data = json.load(f)

        # Extract true labels and predictions
        y_true = [entry for entry in data['gt_result_lst']]
        y_pred = [entry for entry in data['results']]

        # Filter to kept labels only
        mask_true = [l in kept_labels for l in y_true]
        mask_pred = [l in kept_labels for l in y_pred]
        mask_both = [m1 and m2 for m1, m2 in zip(mask_true, mask_pred)]
        y_true_filtered = [y_true[i] for i in range(len(y_true)) if mask_both[i]]
        y_pred_filtered = [y_pred[i] for i in range(len(y_pred)) if mask_both[i]]

        # Remap labels to 0-indexed for sklearn
        label_map = {l: i for i, l in enumerate(kept_labels)}
        y_true_mapped = [label_map[l] for l in y_true_filtered]
        y_pred_mapped = [label_map[l] for l in y_pred_filtered]

        # Calculate metrics
        acc = accuracy_score(y_true_mapped, y_pred_mapped)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true_mapped, y_pred_mapped, labels=list(range(len(kept_labels))), zero_division=0
        )
        conf_mat = confusion_matrix(y_true_mapped, y_pred_mapped, labels=list(range(len(kept_labels))))

        # Create output folder
        exp_name = json_path.split('/')[-1].replace('.json', '')
        output_dir = os.path.join(output_base, exp_name)
        os.makedirs(output_dir, exist_ok=True)

        # Save metrics table to CSV
        metrics_df = pd.DataFrame({
            'label': kept_label_names,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })
        metrics_df.loc['overall'] = ['overall', acc, acc, (2*precision*recall/(precision+recall+1e-8)).mean()]
        metrics_csv_path = os.path.join(output_dir, 'metrics.csv')
        metrics_df.to_csv(metrics_csv_path, index=False)
        print(f"Saved metrics to {metrics_csv_path}")

        # Save confusion matrix plot
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
                    xticklabels=kept_label_names, yticklabels=kept_label_names, ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title(f'Confusion Matrix - {exp_name}')
        plt.tight_layout()
        conf_mat_path = os.path.join(output_dir, 'confusion_matrix.pdf')
        fig.savefig(conf_mat_path, format='pdf')
        plt.close()
        print(f"Saved confusion matrix to {conf_mat_path}")

        # Store summary
        results_summary.append({
            'experiment': exp_name,
            'type': exp_type,
            'accuracy': acc,
            'avg_precision': precision.mean(),
            'avg_recall': recall.mean(),
            'avg_f1': f1.mean()
        })

# Save overall summary
summary_df = pd.DataFrame(results_summary)
summary_csv = os.path.join(output_base, 'summary.csv')
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved summary to {summary_csv}")
print(summary_df)

Saved metrics to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta/office0_lcx_1_cross_user/metrics.csv
Saved confusion matrix to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta/office0_lcx_1_cross_user/confusion_matrix.pdf
Saved metrics to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta/office0_sf_0_cross_user/metrics.csv
Saved confusion matrix to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta/office0_sf_0_cross_user/confusion_matrix.pdf
Saved metrics to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta/hall5_cross_env/metrics.csv
Saved confusion matrix to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/output/meta/hall5_cross_env/confusion_matrix.pdf
Saved metrics to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detect